# 1 — Preprocessing: splits, clips, shared folder

Run this **once**, and only on the machine that holds the DOLOS annotations. Everything
happens here: splits, download, audio, `dataset.csv`. Nobody else needs to run it.

| | |
|---|---|
| **Time** | 15 s for the splits, 2–4 h for the download |
| **Writes** | `splits/`, `clips/`, `audio/`, `dataset.csv` — directly into the shared Drive folder |
| **Resumable** | yes: re-running skips what is already there |

Writing straight to Drive costs about a second per file, but a disconnect no longer throws
away three hours of work: what is on Drive stays, and the next run continues from there.

## 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.isdir("/content/DeepLearningCourse/.git"):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /content/DeepLearningCourse
%cd /content/DeepLearningCourse/lie-detector
!git pull -q
import sys; sys.path.insert(0, '.')

In [ ]:
# Colab has pandas, ffmpeg and tqdm; these two it lacks.
%pip install -q yt-dlp openpyxl tqdm

In [ ]:
from pathlib import Path

# The shared folder. Add a Drive shortcut named "dolos" in your My Drive if a
# teammate owns it: this path then resolves identically for everybody.
DRIVE = Path("/content/drive/MyDrive/dolos")
for sub in ("clips", "audio", "splits"):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
print(DRIVE, "->", sorted(p.name for p in DRIVE.iterdir()))

## 1 — Annotations

The DOLOS spreadsheets are not in git (licence) and only this notebook needs them.
If they are already in `dataset/DOLOS/` this cell does nothing; otherwise it extracts
`dolos_annotations.zip` from the shared folder.

In [ ]:
import zipfile
from pathlib import Path

DOLOS = Path("dataset/DOLOS")
archive = DRIVE / "dolos_annotations.zip"

if (DOLOS / "Dolos.xlsx").exists():
    print("annotations already in place")
elif archive.exists():
    print("extracting", archive.name)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall("dataset")
else:
    raise SystemExit(
        "No DOLOS annotations: upload dolos_annotations.zip to the shared folder first."
    )
print(sorted(p.name for p in DOLOS.iterdir()))

## 2 — Download the clips

Cuts 1,671 clips, extracts the audio, writes `dataset.csv` with the 41 MUMIN columns.
Four workers is the sweet spot; if YouTube throttles you, stop the cell, drop to `--workers 2`
and run it again — it resumes from what is already on Drive.

Two progress indicators: the `download` bar (speed, ok/err counts) and, from this cell, one
line per minute with the real number of clips sitting on Drive plus an ETA.

In [ ]:
import subprocess, sys, time
sys.path.insert(0, "scripts")
from build_dataset import load_clips

TOTAL = sum(1 for clip in load_clips(Path("dataset/DOLOS")) if clip.duration > 0)
initial = len(list((DRIVE / "clips").glob("*.mp4")))
print(f"{TOTAL} clips expected, {initial} already on Drive")

cmd = [sys.executable, "scripts/build_dataset.py", "--workers", "4", "--quiet",
       "--out-dir", str(DRIVE / "clips"),
       "--audio-dir", str(DRIVE / "audio"),
       "--csv", str(DRIVE / "dataset.csv")]

start, process = time.time(), subprocess.Popen(cmd)
try:
    while process.poll() is None:
        done = len(list((DRIVE / "clips").glob("*.mp4")))
        minutes = (time.time() - start) / 60
        rate = (done - initial) / minutes if minutes > 0.5 else 0.0
        eta = f"{(TOTAL - done) / rate:5.0f}" if rate > 0 else "   --"
        print(f"{done:5d}/{TOTAL}  {done / TOTAL * 100:5.1f}%  "
              f"{rate:5.1f} clip/min  ETA {eta} min", flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    process.terminate()
    raise
print("exit code:", process.poll())
print("clips on Drive:", len(list((DRIVE / "clips").glob("*.mp4"))))

In [ ]:
# What the run actually did, and why the rest failed.
import pandas as pd

table = pd.read_csv(DRIVE / "dataset.csv")
print(table.status.value_counts().to_string())
failed = table[table.status == "error"]
if len(failed):
    print(f"\n{len(failed)} clips failed, first five:")
    for row in failed.head(5).itertuples():
        print(f"  {row.clip_id}: {str(row.error)[:110]}")
    print("\nre-run the cell above to retry them; if they all look like the same "
          "YouTube error, lower --workers to 2")

In [ ]:
import pandas as pd

table = pd.read_csv(DRIVE / "dataset.csv")
print(table.status.value_counts().to_string())
print()
ok = int((table.status == "ok").sum())
print(f"{len(table)} annotated -> {ok} usable ({ok / len(table) * 100:.1f}%)")
print("this is the number to write in the report")

## 3 — Freeze the splits

Built **after** the download, on the clips that actually exist: a protocol that references a missing clip
silently shrinks every fold, and two members would end up training on different subsets.

It reserves 15% of the subjects as test and cuts the rest into 5 folds, with a validation slice inside each.
Written straight into `splits/` on Drive, so everybody reads the same files.

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "scripts/make_splits.py",
    "--out-dir", str(DRIVE / "splits"),
    "--available-from", str(DRIVE / "dataset.csv"),
    "--test-fraction", "0.15",
    "--folds", "5",
    "--val-fraction", "0.15",
], check=True)
print(sorted(p.name for p in (DRIVE / "splits").iterdir()))

## 4 — Verify, then tell the group

The last check before everybody starts: five folds, the composition of each split, and the
fingerprint of the table.

In [ ]:
subprocess.run([sys.executable, "-m", "lib.dataset", "--data-dir", str(DRIVE)], check=False)

In [ ]:
import hashlib

digest = hashlib.sha256((DRIVE / "dataset.csv").read_bytes()).hexdigest()[:12]
print(f"dataset.csv sha256:{digest}")
print("clips on drive:", len(list((DRIVE / "clips").glob("*"))))
print("audio on drive:", len(list((DRIVE / "audio").glob("*"))))

Post in the chat: **how many clips are `ok`** and the **sha256 prefix** of `dataset.csv`.
Everybody else opens `architecture_template.ipynb`, points it at the same folder, and never
touches DOLOS.

Share the folder **view-only** with the other three, with write access only on
`cached_features/`: the splits and `dataset.csv` are the protocol, and a coherent
regeneration would not be caught by the hash check.